In [2]:
start_month = "2025-01"
end_month = "2025-06"

StatementMeta(, b6b99413-ebf5-4464-a497-78c9df3f1700, 3, Finished, Available, Finished, False)

In [1]:
df = spark.read.parquet("Files/taxi/yellow/")
df.printSchema()
print(df.count())
display(df.limit(20))

StatementMeta(, 0247b47e-d30a-4b8f-a025-4a7a7bc9416c, 3, Finished, Available, Finished, False)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

24083384


SynapseWidget(Synapse.DataFrame, 77676ba5-9e75-4591-9048-5f30f03f39a5)

In [2]:
import pyspark.sql.functions as F
df.select(F.min(df.tpep_pickup_datetime)).show()
df.select(F.max(df.tpep_pickup_datetime)).show()
display(df.filter(df.fare_amount <0).limit(10))
display(df.filter(df.trip_distance == 0).limit(10))
display(df.filter(df.passenger_count.isNull()))

StatementMeta(, 0247b47e-d30a-4b8f-a025-4a7a7bc9416c, 4, Finished, Available, Finished, False)

+-------------------------+
|min(tpep_pickup_datetime)|
+-------------------------+
|      2007-12-05 18:45:00|
+-------------------------+

+-------------------------+
|max(tpep_pickup_datetime)|
+-------------------------+
|      2025-06-30 23:59:59|
+-------------------------+



SynapseWidget(Synapse.DataFrame, a19f5330-0b7d-4cbb-9d60-9b4403f9b39c)

SynapseWidget(Synapse.DataFrame, 03faaed2-a431-4a4c-a4da-c9099676f581)

SynapseWidget(Synapse.DataFrame, e8a91b66-0325-4a72-ab52-70f15eb761b3)

In [3]:
df = (
    df
    .withColumnRenamed("VendorID", "vendor_id")
    .withColumnRenamed("tpep_pickup_datetime", "pickup_datetime")
    .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")
    .withColumnRenamed("RatecodeID", "ratecode_id")
    .withColumnRenamed("PULocationID", "pu_location_id")
    .withColumnRenamed("DOLocationID", "do_location_id")
    .withColumnRenamed("Airport_fee", "airport_fee")
)

df = (
    df
    .withColumn("pickup_date", F.to_date(F.col("pickup_datetime")))
    .withColumn("pickup_hour", F.hour(F.col("pickup_datetime")))
    .withColumn(
        "trip_duration_min",
        (F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")) / 60
    )
)

before = df.count()

df_clean = df.filter(
    (F.date_format("pickup_datetime", "yyyy-MM") >= start_month) &
    (F.date_format("pickup_datetime", "yyyy-MM") <= end_month) &
    (F.col("trip_duration_min") >= 0) &
    (F.col("trip_duration_min") <= 1440) &
    (F.col("trip_distance") <= 200)
)

after = df_clean.count()

print(before, after, round((before - after) / before * 100, 3), "% dropped")

StatementMeta(, 0247b47e-d30a-4b8f-a025-4a7a7bc9416c, 5, Finished, Available, Finished, False)

24083384 24081200 0.009 % dropped


In [4]:
neg = df_clean.filter(F.col("total_amount") < 0)
pos = df_clean.filter(F.col("total_amount") > 0)

matched = neg.alias("n").join(
    pos.alias("p"),
    (F.col("n.vendor_id") == F.col("p.vendor_id")) &
    (F.col("n.pickup_datetime") == F.col("p.pickup_datetime")) &
    (F.col("n.dropoff_datetime") == F.col("p.dropoff_datetime")) &
    (F.col("n.pu_location_id") == F.col("p.pu_location_id")) &
    (F.col("n.do_location_id") == F.col("p.do_location_id")) &
    (F.col("n.total_amount") == -F.col("p.total_amount")),
    "inner"
)

print("negative rows:", neg.count())
print("matched pairs:", matched.count())

StatementMeta(, 0247b47e-d30a-4b8f-a025-4a7a7bc9416c, 6, Finished, Available, Finished, False)

negative rows: 446291
matched pairs: 359940


In [5]:
df_silver = (
    df_clean
    .withColumn("is_refund", F.col("total_amount") < 0)
    .withColumn("is_zero_distance", F.col("trip_distance") == 0)
    .withColumn("is_zero_duration", F.col("trip_duration_min") == 0)
    .withColumn(
        "is_unknown_zone",
        (F.col("pu_location_id").isin(264, 265)) |
        (F.col("do_location_id").isin(264, 265))
    )
    .withColumn("is_long_trip", F.col("trip_duration_min") > 180)
    .withColumn(
        "has_any_issue",
        F.col("is_refund") |
        F.col("is_zero_distance") |
        F.col("is_zero_duration") |
        F.col("is_unknown_zone") |
        F.col("is_long_trip")
    )
)

StatementMeta(, 0247b47e-d30a-4b8f-a025-4a7a7bc9416c, 7, Finished, Available, Finished, False)

In [6]:
df_silver.select(
    F.count("*").alias("total_rows"),
    F.sum(F.col("is_refund").cast("int")).alias("is_refund"),
    F.sum(F.col("is_zero_distance").cast("int")).alias("is_zero_distance"),
    F.sum(F.col("is_zero_duration").cast("int")).alias("is_zero_duration"),
    F.sum(F.col("is_unknown_zone").cast("int")).alias("is_unknown_zone"),
    F.sum(F.col("is_long_trip").cast("int")).alias("is_long_trip"),
    F.sum(F.col("has_any_issue").cast("int")).alias("has_any_issue")
).show()

StatementMeta(, 0247b47e-d30a-4b8f-a025-4a7a7bc9416c, 8, Finished, Available, Finished, False)

+----------+---------+----------------+----------------+---------------+------------+-------------+
|total_rows|is_refund|is_zero_distance|is_zero_duration|is_unknown_zone|is_long_trip|has_any_issue|
+----------+---------+----------------+----------------+---------------+------------+-------------+
|  24081200|   446291|          661806|          196296|         178595|        8603|      1431414|
+----------+---------+----------------+----------------+---------------+------------+-------------+



In [7]:
df_silver.write.mode("overwrite").format("delta").saveAsTable("lh_silver.dbo.taxi_trips")

StatementMeta(, 0247b47e-d30a-4b8f-a025-4a7a7bc9416c, 9, Finished, Available, Finished, False)

In [8]:
lk = spark.read.option("header", True).option("inferSchema", True).csv("Files/taxi/lookup/taxi_zone_lookup.csv")
lk.show(10)
lk.printSchema()
lk = (lk
        .withColumnRenamed("LocationID", "location_id")
        .withColumnRenamed("Borough", "borough")
        .withColumnRenamed("Zone", "zone")
        )

lk.printSchema()
lk.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("lh_silver.dbo.taxi_zones")


StatementMeta(, 0247b47e-d30a-4b8f-a025-4a7a7bc9416c, 10, Finished, Available, Finished, False)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 10 rows
root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true